In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats import kurtosis, skew
from scipy.fft import rfft, rfftfreq
from datetime import datetime

# --- 1. ПОМОЩНИК ДЛЯ РАСЧЕТА ПРИЗНАКОВ СИГНАЛА ---
def get_signal_stats(sig, fs=20000):
    """Считает временные и частотные признаки для одного канала"""
    stats = {}
    
    # --- Временные признаки ---
    rms = np.sqrt(np.mean(sig**2))
    stats['rms'] = rms
    stats['mean'] = np.mean(sig)
    stats['var'] = np.var(sig)
    stats['kurt'] = kurtosis(sig)
    stats['skew'] = skew(sig)
    stats['crest'] = np.max(np.abs(sig)) / rms if rms > 0 else 0
    stats['p2p'] = np.ptp(sig)
    stats['energy'] = np.sum(sig**2)
    
    # --- Частотные признаки (FFT) ---
    n = len(sig)
    abs_fft = np.abs(rfft(sig))
    freqs = rfftfreq(n, 1/fs)
    
    # Доминирующая частота и спектральный центроид
    stats['dom_freq'] = freqs[np.argmax(abs_fft)]
    stats['spec_centroid'] = np.sum(freqs * abs_fft) / np.sum(abs_fft)
    
    # Полосы частот (согласно ТЗ)
    stats['band_0_2k'] = np.sum(abs_fft[(freqs >= 0) & (freqs < 2000)])
    stats['band_2_5k'] = np.sum(abs_fft[(freqs >= 2000) & (freqs < 5000)])
    stats['band_5_10k'] = np.sum(abs_fft[(freqs >= 5000) & (freqs <= 10000)])
    
    return stats

# --- 2. ОСНОВНАЯ ФУНКЦИЯ ОБХОДА ФАЙЛОВ ---
def extract_advanced_features(data_path):
    features = []
    files = sorted(os.listdir(data_path))
    print(f"📦 Обработка {len(files)} файлов...")
    
    for filename in files:
        try:
            file_path = os.path.join(data_path, filename)
            df = pd.read_csv(file_path, sep='\t', header=None)
            
            # Время из названия
            dt = datetime.strptime(filename, '%Y.%m.%d.%H.%M.%S')
            file_features = {'timestamp': dt}
            
            # Проходим по подшипникам (колонки)
            for i in range(df.shape[1]):
                sig = df[i].values
                # Вызываем нашего помощника
                sig_stats = get_signal_stats(sig)
                
                # Добавляем префикс b1_, b2_ и т.д. к каждому ключу
                prefix = f'b{i+1}_'
                for stat_name, value in sig_stats.items():
                    file_features[prefix + stat_name] = value
                
            features.append(file_features)
        except Exception as e:
            print(f"⚠️ Ошибка в {filename}: {e}")
            continue
        
    res_df = pd.DataFrame(features).sort_values('timestamp').reset_index(drop=True)
    return res_df

# --- 3. ГРУППОВЫЕ ПРИЗНАКИ (ПОСЛЕ СБОРА ВСЕХ ДАННЫХ) ---
def add_group_features(df):
    """Считает корреляцию и относительные показатели между подшипниками"""
    # Список колонок с RMS
    rms_cols = [c for c in df.columns if '_rms' in c]
    
    # Среднее по системе
    df['system_rms_mean'] = df[rms_cols].mean(axis=1)
    
    # Насколько каждый подшипник отклоняется от среднего (в %)
    for col in rms_cols:
        b_name = col.split('_')[0]
        df[f'{b_name}_noise_ratio'] = df[col] / df['system_rms_mean']
        
    return df

In [2]:
import pandas as pd
import numpy as np
import os
from scipy.stats import kurtosis, skew
from datetime import datetime

def extract_advanced_features(data_path):
    features = []
    files = sorted(os.listdir(data_path))
    print(f"📦 Обработка {len(files)} файлов...")
    
    for filename in files:
        try:
            # 1. Читаем файл
            file_path = os.path.join(data_path, filename)
            df = pd.read_csv(file_path, sep='\t', header=None)
            
            # 2. Извлекаем время из названия файла (критично для RUL!)
            # Формат: 2004.02.12.10.32.39
            dt = datetime.strptime(filename, '%Y.%m.%d.%H.%M.%S')
            
            file_features = {'timestamp': dt}
            
            # 3. Считаем признаки для каждого канала (динамически)
            for i in range(df.shape[1]):
                sig = df[i].values
                rms = np.sqrt(np.mean(sig**2))
                
                prefix = f'b{i+1}'
                file_features[f'{prefix}_rms'] = rms
                file_features[f'{prefix}_kurt'] = kurtosis(sig)
                file_features[f'{prefix}_crest'] = np.max(np.abs(sig)) / rms if rms > 0 else 0
                file_features[f'{prefix}_skew'] = skew(sig)
                file_features[f'{prefix}_p2p'] = np.ptp(sig) # Peak-to-Peak
                
            features.append(file_features)
        except Exception as e:
            # Выводим ошибку, если файл битый (в NASA IMS такое бывает)
            print(f"⚠️ Ошибка в файле {filename}: {e}")
            continue
        
    # Создаем DF и сразу сортируем по времени
    res_df = pd.DataFrame(features).sort_values('timestamp').reset_index(drop=True)
    return res_df

# Использование
# DATA_PATH = '2nd_test'
# df_feat = extract_advanced_features(DATA_PATH)